# CI相关音频分类实战

## 学习目标

- 理解"分类"在CI研究中的具体应用场景
- 实战1：搭建语音活动检测(VAD)模型
- 实战2：搭建带噪语音可懂度分类模型
- 掌握模型调优技巧：学习率搜索、混合精度训练、梯度累积

## 1. 分类在CI研究中的应用

1. **语音活动检测(VAD)**：判断当前帧是语音还是噪声。ACE策略中"选通道"的前提。
2. **环境声分类**：帮助CI用户感知周围环境。
3. **可懂度分类**：判断带噪语音对CI用户是否可懂。
4. **电刺激通道选择**：用深度学习替代固定规则选通道。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import os
from torch.utils.data import Dataset, DataLoader

plt.rcParams['font.size'] = 12

SN_DIR = '../data/speech_noise_dataset'
INTEL_DIR = '../data/intelligibility_dataset'

SN_AVAIL = os.path.exists(os.path.join(SN_DIR, 'speech'))
INTEL_AVAIL = os.path.exists(os.path.join(INTEL_DIR, 'intelligible'))

if SN_AVAIL: print('语音/噪声数据集已就绪')
else: print('语音/噪声数据集未找到, 将使用合成数据')
if INTEL_AVAIL: print('可懂度数据集已就绪')
else: print('可懂度数据集未找到, 将使用合成数据')

## 2. 实战1：语音活动检测(VAD)

VAD: 输入音频帧 -> 语音/非语音

这是CI编码策略的基础：
- 传统ACE：固定规则(能量阈值)判断有无语音
- DL的VAD：学习更鲁棒的判断，在噪声中表现更好

```
音频帧 -> 特征提取 -> VAD模型 -> 语音/非语音
                                    |
                               选通道/不选通道
```

In [ ]:
class VADDataset(Dataset):
    def __init__(self, data_dir=None, sr=16000, n_mels=64, seg_dur=1.0,
                 n_samples=None, augment=False):
        self.sr = sr; self.n_mels = n_mels
        self.seg_len = int(sr * seg_dur)
        self.augment = augment
        self.mel_t = T.MelSpectrogram(sr, n_fft=512, hop_length=160, n_mels=n_mels)
        self.data = []; self.labels = []
        if data_dir and os.path.exists(os.path.join(data_dir, 'speech')):
            sp_dir = os.path.join(data_dir, 'speech')
            ns_dir = os.path.join(data_dir, 'noise')
            sp_files = [os.path.join(sp_dir,f) for f in os.listdir(sp_dir) if f.endswith('.wav')]
            ns_files = [os.path.join(ns_dir,f) for f in os.listdir(ns_dir) if f.endswith('.wav')]
            if n_samples:
                sp_files = sp_files[:n_samples//2]
                ns_files = ns_files[:n_samples//2]
            for f in sp_files: self.data.append(f); self.labels.append(1)
            for f in ns_files: self.data.append(f); self.labels.append(0)
        else:
            n_syn = n_samples or 1000
            for i in range(n_syn//2):
                t = np.linspace(0, seg_dur, self.seg_len, endpoint=False)
                freq = np.random.uniform(100, 400)
                sp = sum(np.random.uniform(0.1,0.4)*np.sin(2*np.pi*freq*k*t) for k in range(1, np.random.randint(2,6)))
                sp += 0.02*np.random.randn(self.seg_len)
                self.data.append(torch.FloatTensor(sp)); self.labels.append(1)
                ns = np.random.uniform(0.05,0.3)*np.random.randn(self.seg_len)
                self.data.append(torch.FloatTensor(ns)); self.labels.append(0)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        label = self.labels[idx]; raw = self.data[idx]
        if isinstance(raw, str):
            wav, orig_sr = torchaudio.load(raw)
            if orig_sr != self.sr: wav = T.Resample(orig_sr, self.sr)(wav)
            if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
            wav = wav.squeeze(0)
            if len(wav) > self.seg_len:
                s = np.random.randint(0, len(wav)-self.seg_len); wav = wav[s:s+self.seg_len]
            elif len(wav) < self.seg_len:
                wav = torch.nn.functional.pad(wav, (0, self.seg_len-len(wav)))
        else: wav = raw
        if self.augment and np.random.random() < 0.3:
            wav = wav + np.random.uniform(0.01,0.1)*torch.randn_like(wav)
        mel = self.mel_t(wav.unsqueeze(0))
        mel_db = torchaudio.functional.amplitude_to_DB(mel, top_db=80)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        return mel_db.squeeze(0), label

vad_train = VADDataset(SN_DIR if SN_AVAIL else None, n_samples=1000, augment=True)
vad_val = VADDataset(SN_DIR if SN_AVAIL else None, n_samples=400)
print('训练集:', len(vad_train), '验证集:', len(vad_val))

In [ ]:
class VADCNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
        )
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(64, n_classes))
    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)
        f = self.features(x).view(x.size(0),-1)
        return self.classifier(f)

model = VADCNN()
x = torch.randn(4, 64, 100)
print('输入:', x.shape, '输出:', model(x).shape)

In [ ]:
# 训练VAD
train_loader = DataLoader(vad_train, batch_size=32, shuffle=True)
val_loader = DataLoader(vad_val, batch_size=32)
vad_model = VADCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vad_model.parameters(), lr=0.001)

train_losses, val_accs = [], []
for epoch in range(20):
    vad_model.train(); total_loss = 0
    for feats, labels in train_loader:
        out = vad_model(feats); loss = criterion(out, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    train_losses.append(total_loss/len(train_loader))
    vad_model.eval(); correct = total = 0
    with torch.no_grad():
        for feats, labels in val_loader:
            correct += (vad_model(feats).argmax(1)==labels).sum().item()
            total += labels.size(0)
    val_accs.append(correct/total)
    if (epoch+1)%5==0: print('Epoch %2d, Loss: %.4f, Val Acc: %.4f' % (epoch+1, train_losses[-1], val_accs[-1]))

print('最终VAD准确率: %.4f' % val_accs[-1])

### VAD与CI编码策略的连接

VAD输出直接决定ACE策略是否选通道：
```
输入音频 -> VAD -> 有语音? -> 是 -> ACE选通道
                      |-> 否 -> 不刺激(省电)
```
传统ACE用能量阈值做VAD，噪声中容易误判。DL的VAD更鲁棒。

## 3. 实战2：带噪语音可懂度分类

可懂度分类：输入带噪语音 -> 可懂/不可懂

这对接CI可懂度研究——可以作为语音增强效果的评价工具。

In [ ]:
class IntelligibilityDataset(Dataset):
    def __init__(self, data_dir=None, sr=16000, n_mels=64, n_samples=None):
        self.sr = sr; self.n_mels = n_mels
        self.mel_t = T.MelSpectrogram(sr, n_fft=512, hop_length=160, n_mels=n_mels)
        self.data = []; self.labels = []
        if data_dir and os.path.exists(os.path.join(data_dir, 'intelligible')):
            intel_dir = os.path.join(data_dir, 'intelligible')
            unintel_dir = os.path.join(data_dir, 'unintelligible')
            intel_f = [os.path.join(intel_dir,f) for f in os.listdir(intel_dir) if f.endswith('.wav')]
            unintel_f = [os.path.join(unintel_dir,f) for f in os.listdir(unintel_dir) if f.endswith('.wav')]
            if n_samples:
                intel_f = intel_f[:n_samples//2]
                unintel_f = unintel_f[:n_samples//2]
            for f in intel_f: self.data.append(f); self.labels.append(1)
            for f in unintel_f: self.data.append(f); self.labels.append(0)
        else:
            n_syn = n_samples or 800
            for i in range(n_syn//2):
                t = np.linspace(0, 1.0, sr, endpoint=False)
                freq = np.random.uniform(100, 400)
                sp = sum(np.random.uniform(0.1,0.3)*np.sin(2*np.pi*freq*k*t) for k in range(1,4))
                self.data.append(torch.FloatTensor(sp + 0.05*np.random.randn(len(t)))); self.labels.append(1)
                self.data.append(torch.FloatTensor(sp + 0.8*np.random.randn(len(t)))); self.labels.append(0)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        label = self.labels[idx]; raw = self.data[idx]
        if isinstance(raw, str):
            wav, orig_sr = torchaudio.load(raw)
            if orig_sr != self.sr: wav = T.Resample(orig_sr, self.sr)(wav)
            if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
            wav = wav.squeeze(0)
        else: wav = raw
        mel = self.mel_t(wav.unsqueeze(0))
        mel_db = torchaudio.functional.amplitude_to_DB(mel, top_db=80)
        mel_db = (mel_db - mel_db.mean())/(mel_db.std()+1e-8)
        return mel_db.squeeze(0), label

intel_train = IntelligibilityDataset(INTEL_DIR if INTEL_AVAIL else None, n_samples=800)
intel_val = IntelligibilityDataset(INTEL_DIR if INTEL_AVAIL else None, n_samples=300)
print('训练集:', len(intel_train), '验证集:', len(intel_val))

In [ ]:
# 训练可懂度分类器
train_loader2 = DataLoader(intel_train, batch_size=32, shuffle=True)
val_loader2 = DataLoader(intel_val, batch_size=32)

intel_model = VADCNN(n_mels=64, n_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(intel_model.parameters(), lr=0.001)

train_losses2, val_accs2 = [], []
for epoch in range(15):
    intel_model.train(); total_loss = 0
    for feats, labels in train_loader2:
        out = intel_model(feats); loss = criterion(out, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    train_losses2.append(total_loss/len(train_loader2))
    intel_model.eval(); correct = total = 0
    with torch.no_grad():
        for feats, labels in val_loader2:
            correct += (intel_model(feats).argmax(1)==labels).sum().item()
            total += labels.size(0)
    val_accs2.append(correct/total)
    if (epoch+1)%5==0: print('Epoch %2d, Loss: %.4f, Val Acc: %.4f' % (epoch+1, train_losses2[-1], val_accs2[-1]))

print('最终可懂度分类准确率: %.4f' % val_accs2[-1])

## 4. 模型调优技巧

In [ ]:
# 技巧1: 学习率搜索 (LR Range Test)
def lr_range_test(model, loader, criterion, lr_start=1e-6, lr_end=1.0, n_steps=100):
    model.train()
    lrs = np.logspace(np.log10(lr_start), np.log10(lr_end), n_steps)
    losses = []
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_start)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: lrs[step]/lr_start)
    data_iter = iter(loader)
    for step in range(n_steps):
        try: feats, labels = next(data_iter)
        except StopIteration: data_iter = iter(loader); feats, labels = next(data_iter)
        optimizer.zero_grad()
        out = model(feats); loss = criterion(out, labels)
        loss.backward(); optimizer.step(); scheduler.step()
        losses.append(loss.item())
        if step > 0 and losses[-1] > 4*min(losses): break
    return lrs[:len(losses)], losses

test_model = VADCNN()
lrs, losses = lr_range_test(test_model, train_loader, nn.CrossEntropyLoss())
plt.figure(figsize=(10, 4))
plt.semilogx(lrs, losses); plt.xlabel('Learning Rate'); plt.ylabel('Loss')
plt.title('LR Range Test'); plt.grid(True, alpha=0.3); plt.show()
print('选择loss下降最快区域的中点作为学习率')

In [ ]:
# 技巧2: 混合精度训练
def train_amp(model, loader, criterion, epochs=5, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda')
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for feats, labels in loader:
            if torch.cuda.is_available(): feats, labels = feats.cuda(), labels.cuda()
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(feats); loss = criterion(out, labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item()
        if (epoch+1)%2==0: print('Epoch %d, Loss: %.4f' % (epoch+1, total_loss/len(loader)))

if torch.cuda.is_available():
    print('GPU可用, 运行混合精度训练...')
    train_amp(VADCNN().cuda(), train_loader, nn.CrossEntropyLoss())
else:
    print('GPU不可用, 跳过(在Colab/服务器上运行)')

In [ ]:
# 技巧3: 梯度累积 (等效batch_size = 物理batch * accum_steps)
def train_grad_accum(model, loader, criterion, epochs=5, lr=0.001, accum_steps=4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad(); total_loss = 0
        for i, (feats, labels) in enumerate(loader):
            out = model(feats); loss = criterion(out, labels) / accum_steps
            loss.backward()
            if (i+1) % accum_steps == 0:
                optimizer.step(); optimizer.zero_grad()
            total_loss += loss.item() * accum_steps
        if (epoch+1)%2==0: print('Epoch %d, Loss: %.4f' % (epoch+1, total_loss/len(loader)))

print('梯度累积演示 (等效batch_size=32*4=128)')
train_grad_accum(VADCNN(), train_loader, nn.CrossEntropyLoss(), accum_steps=4)

### 调优技巧总结

| 技巧 | 作用 | 适用场景 |
|------|------|----------|
| LR Range Test | 找最佳学习率范围 | 训练初期 |
| 混合精度训练 | 减少GPU内存+加速2x | GPU训练 |
| 梯度累积 | 模拟大batch_size | GPU内存不够时 |
| 早停 | 防止过拟合 | 任何训练 |

租用服务器策略：混合精度+梯度累积，在T4上训练更大batch，减少租用时长。

## 本节要点

1. VAD是CI编码策略的基础环节——DL方法在噪声中更鲁棒
2. 可懂度分类可作为语音增强/编码策略的评价工具
3. VAD与ACE策略的连接：VAD决定"是否选通道"
4. 调优技巧在GPU资源有限时至关重要
5. 所有技巧在Colab的T4 GPU上都可以使用

---
返回目录：[README.md](../README.md)